# R7.6｜四类基线公平比较：hybrid deep-learning prediction

**冻结研究合同：R7.5 的 10 个滚动年份、每年 2 个种子、180 天历史、提前 30/60/90 天、逐折 train-only EOF，逐年结果不按测试表现筛选。**

基线：① Persistence（持续性）② Climatology（训练期日历气候态）③ AR-EOF（训练期 EOF 主成分的多滞后 Ridge-AR）④ Parameter-matched LSTM（两分支原始特征输入、参数量配对、相同物理损失与训练预算）。所有结果以 **GLORYS12V1 50 m 派生流轴**为真值；本轮不将 ADT 数据混入后报标签。

> 对于非神经基线，不虚构“训练 Epoch”。AR-EOF 只在 R7.5 事先划定的验证年份选择 lag/alpha。对照主模型本身不做任何重新训练。2014/2019 先前已参与研发，单独列出敏感性统计。

In [ ]:
# 0. 配置、环境、合同（更改ROOT后直接 Run All）
from dataclasses import dataclass,asdict
from pathlib import Path
from datetime import datetime,timezone
from io import BytesIO
import hashlib,json,math,os,random,time,zipfile,gc,traceback,warnings,sys
import numpy as np,pandas as pd
from scipy.spatial import cKDTree
from scipy.linalg import cho_factor,cho_solve
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
import matplotlib.pyplot as plt
from IPython.display import display,FileLink

PAPER_NAME='hybrid deep-learning prediction'
B_NAMES=('Persistence','Climatology','AR-EOF','Parameter-matched LSTM')
@dataclass
class Config:
    ROOT:str='/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL:str='final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    R75_REL:str='R7_5_HybridDeepLearning_MultiYearRollingHindcast'
    R75_ZIP:str='R75_上传给ChatGPT_多年滚动后报_hybrid_deep-learning_prediction.zip'
    OUT_REL:str='R7_6_FairBaselines_2026'
    HISTORY:int=180
    LEADS:tuple=(30,60,90)
    YEARS:tuple=(2005,2008,2011,2014,2017,2019,2021,2022,2023,2024)
    SEEDS:tuple=(71,72)
    MAX_INPUT_GAP:int=2
    MAX_MISSING_FRAC:float=.03
    DROP_BRANCH1_FIRST:int=12
    BRANCH2_MAX_LON:float=145.
    BATCH_SIZE:int=32
    DROPOUT:float=.30
    ALPHA:float=.40
    KM_SCALE:float=25.
    HUBER_BETA:float=.50
    LR:float=.001
    WEIGHT_DECAY:float=.0001
    EPOCHS:int=36
    PATIENCE:int=8
    USE_AMP:bool=True
    NUM_WORKERS:int=0
    AR_LAGS:tuple=(3,7,14,30)
    AR_ALPHAS:tuple=(.1,10.,1000.)
    AR_INPUT_MAX_PCS:int=32
    CLIM_WINDOW_DAYS:int=15
    BOOT_REPS:int=400
    BOOT_BLOCK_DAYS:int=30
    FORCE_RETRAIN:bool=False
    PREFLIGHT_ONLY:bool=False
    RUN_FULL:bool=True
    FIG_DPI:int=240

c=Config()
ROOT=Path(c.ROOT);DS=ROOT/c.DATASET_REL;R75=ROOT/c.R75_REL
R75ZIP=ROOT/c.R75_ZIP
RUN=ROOT/c.OUT_REL;TB=RUN/'tables';WEIGHTS=RUN/'weights';PRED=RUN/'predictions';REP=RUN/'reports';FIG=RUN/'figures_en'
for p in [RUN,TB,WEIGHTS,PRED,REP,FIG]:p.mkdir(parents=True,exist_ok=True)
ARCHIVE=ROOT/'R76_上传给ChatGPT_四类基线公平比较.zip'
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(8<<20),b''):h.update(b)
    return h.hexdigest()

def get_r75_bytes(relative):
    local=R75/relative
    if local.is_file():return local.read_bytes()
    if R75ZIP.is_file():
        with zipfile.ZipFile(R75ZIP) as z:
            if relative in z.namelist():return z.read(relative)
    raise FileNotFoundError('缺少 R7.5 文件：'+relative+'；请保留 R7.5 输出目录或放入其完整ZIP')

def read_r75_csv(relative):return pd.read_csv(BytesIO(get_r75_bytes(relative)))

if not DS.is_file():raise FileNotFoundError('缺少冻结日数据：'+str(DS))
DATASET_SHA=sha256_file(DS)
R75_CONTRACT=json.loads(get_r75_bytes('reports/R75_冻结实验合同.json').decode('utf-8'))
R75_DONE=json.loads(get_r75_bytes('reports/R75_completed_status.json').decode('utf-8'))
assert R75_DONE['complete'] and R75_DONE['completed_fold_seed_runs']==len(c.YEARS)*len(c.SEEDS)
assert DATASET_SHA==R75_CONTRACT['dataset_sha256']==R75_DONE['dataset_sha256'],'DATASET SHA 不一致，严禁比较'
assert tuple(R75_DONE['test_years'])==c.YEARS and tuple(R75_DONE['seeds'])==c.SEEDS,'年份种子不一致'
assert R75_CONTRACT['settings']['HISTORY']==c.HISTORY and tuple(R75_CONTRACT['settings']['LEADS'])==c.LEADS
assert R75_CONTRACT['settings']['DROP_BRANCH1_FIRST']==c.DROP_BRANCH1_FIRST
assert R75_CONTRACT['settings']['BRANCH2_MAX_LON']==c.BRANCH2_MAX_LON
assert R75_CONTRACT['settings']['EPOCHS']==c.EPOCHS and R75_CONTRACT['settings']['PATIENCE']==c.PATIENCE
assert R75_CONTRACT['settings']['BATCH_SIZE']==c.BATCH_SIZE
assert R75_CONTRACT['settings']['LR']==c.LR and R75_CONTRACT['settings']['WEIGHT_DECAY']==c.WEIGHT_DECAY
assert R75_CONTRACT['settings']['ALPHA']==c.ALPHA and R75_CONTRACT['settings']['KM_SCALE']==c.KM_SCALE
assert R75_CONTRACT['settings']['MAX_INPUT_GAP']==c.MAX_INPUT_GAP and R75_CONTRACT['settings']['MAX_MISSING_FRAC']==c.MAX_MISSING_FRAC
assert R75_CONTRACT['paper_model_name']==PAPER_NAME
R75_ISSUES=read_r75_csv('tables/02_all_fold_issue_dates_for_future_baselines.csv')
R75_ERRORS=read_r75_csv('tables/04_all_individual_forecast_errors.csv')
R75_STATUS=read_r75_csv('tables/03_run_status.csv')
assert len(R75_ERRORS)>0
assert R75_ERRORS.model.eq(PAPER_NAME).all()
assert len(R75_ERRORS)==3*len(c.SEEDS)*len(R75_ISSUES[R75_ISSUES.split=='test'])
assert R75_ISSUES[['test_year','split','issue_date']].duplicated().sum()==0
print('DATASET verified',DATASET_SHA[:16], '| Device',DEV, '| R75 issue rows',len(R75_ISSUES),'| R75 test predictions',len(R75_ERRORS))
(REP/'R76_contract.json').write_text(json.dumps({'model_name':PAPER_NAME,'r75_dataset_sha256':DATASET_SHA,'r75_completed':R75_DONE,'settings':asdict(c)},ensure_ascii=False,indent=2),encoding='utf8')

In [ ]:
# 1｜按冻结数据格式恢复双分支/日历，不修改样本或标签
with np.load(DS,allow_pickle=False) as z:
    dates=pd.DatetimeIndex(pd.to_datetime(z['dates'])).normalize()
    b1=np.asarray(z['branch1_lon'],np.float32)
    lat1=np.asarray(z['branch1_target_lats'],float)
    b2=np.asarray(z['branch2_lat'],np.float32)
    lon2=np.asarray(z['branch2_target_lons'],float)
if dates.has_duplicates or not(len(dates)==len(b1)==len(b2)):
    raise RuntimeError('冻结数据重复日期或数量不一致')
order=np.argsort(dates.values);dates=dates[order];b1=b1[order];b2=b2[order]
if not(np.isfinite(b1).all() and np.isfinite(b2).all()):raise RuntimeError('输入冻结数据含NaN/Inf')
b1=b1[:,c.DROP_BRANCH1_FIRST:];lat1=lat1[c.DROP_BRANCH1_FIRST:]
keep2=lon2<=c.BRANCH2_MAX_LON+1e-8;lon2=lon2[keep2];b2=b2[:,keep2]
cal=pd.date_range(dates.min(),dates.max(),freq='D');pos=cal.get_indexer(dates)
if (pos<0).any():raise RuntimeError('时间轴不合法')
valid=np.zeros(len(cal),bool);valid[pos]=True
A1=np.full((len(cal),b1.shape[1]),np.nan,np.float32)
A2=np.full((len(cal),b2.shape[1]),np.nan,np.float32)
A1[pos]=b1;A2[pos]=b2
# 缺失值仅以前向可用历史补齐；未来标签永远取 A1/A2 的真实日样本
F1=pd.DataFrame(A1).ffill(limit=c.MAX_INPUT_GAP).to_numpy(dtype=np.float32)
F2=pd.DataFrame(A2).ffill(limit=c.MAX_INPUT_GAP).to_numpy(dtype=np.float32)
H=int(c.HISTORY); L=max(c.LEADS)
missing=(~valid).astype(np.int32);iss=[]
for i in range(H-1,len(cal)-L):
    if not valid[i]:continue
    win=missing[i-H+1:i+1]
    if win.mean()>c.MAX_MISSING_FRAC:continue
    if np.max(np.diff(np.flatnonzero(np.r_[1,win==0,1]))-1)>c.MAX_INPUT_GAP:continue
    if any(not valid[i+d] for d in c.LEADS):continue
    if not(np.isfinite(F1[i-H+1:i+1]).all() and np.isfinite(F2[i-H+1:i+1]).all()):continue
    iss.append(i)
ISS=np.asarray(iss,np.int32)
if not len(ISS):raise RuntimeError('没有可用180→30/60/90起报日期')
if not all(np.all((cal[ISS+d]-cal[ISS]).days==d) for d in c.LEADS):raise RuntimeError('日期错位')
print('[R76] Frozen dates',len(dates),'range',dates.min().date(),dates.max().date(),'valid anchors:',len(ISS),'B1dim',b1.shape[1],'B2dim',b2.shape[1])

In [ ]:
# 2. 使用 R7.5 现成起报日定义训练/验证/测试；精确检查切分和所有目标日期
R75_FOLDAUDIT=read_r75_csv('tables/01_fold_dates_causal_audit.csv')
FOLDS=[]
index_by_date={str(d.date()):i for i,d in enumerate(cal)}
for y in c.YEARS:
    sub=R75_ISSUES[R75_ISSUES.test_year==y]
    def parse(split):
        df=sub[sub.split==split]
        issues=np.asarray([index_by_date[str(x)] for x in df.issue_date],np.int32)
        for lead in c.LEADS:
            target=np.asarray([str(cal[i+lead].date()) for i in issues]);given=df[f'target_{lead}'].astype(str).to_numpy()
            if not np.array_equal(target,given):raise RuntimeError(f'{y} {split} +{lead} 日期错位')
        return issues
    tr,va,te=[parse(q) for q in ('train','validation','test')]
    assert set(np.r_[tr,va,te]).issubset(set(ISS))
    assert np.all(np.diff(tr)>=1) and np.all(np.diff(va)>=1) and np.all(np.diff(te)>=1)
    assert cal[tr+90].max()<cal[va].min()-pd.Timedelta(days=7)
    assert cal[va+90].max()<cal[te].min()-pd.Timedelta(days=7)
    cut=pd.Timestamp(y-1,1,1)-pd.Timedelta(days=7)
    expected=R75_FOLDAUDIT[R75_FOLDAUDIT.test_year==y].iloc[0]
    assert (len(tr),len(va),len(te))==(expected.train_n,expected.validation_n,expected.test_n)
    assert str(cal[tr[-1]+90].date())==str(expected.train_last_target_90d)
    assert str(cal[va[-1]+90].date())==str(expected.validation_last_target_90d)
    FOLDS.append({'year':y,'train':tr,'val':va,'test':te,'train_cut':cut})
print('[PASS] Strict exact same R7.5 issues (train/val/test) and causal targets for',len(FOLDS),'folds')

In [ ]:
# 3. 原样加载R7.5训练期EOF快照（绝不重新拟合、更不使用未来日期）
class FrozenEOF:
    def __init__(self,mu,sd,V):
        self.mu=np.asarray(mu,np.float64)
        self.sd=np.asarray(sd,np.float64)
        self.V=np.asarray(V,np.float32)
        self.n=self.V.shape[1]
    def z(self,x):return ((np.asarray(x)-self.mu)/self.sd).astype(np.float32)
    def pc(self,x):return (((np.asarray(x)-self.mu)/self.sd)@self.V).astype(np.float32)
    def invert(self,p):return ((np.asarray(p)@self.V.T)*self.sd+self.mu).astype(np.float32)

def make_fold_data(f):
    raw=get_r75_bytes(f"trainonly_eof/trainonly_EOF_{f['year']}.npz")
    with np.load(BytesIO(raw),allow_pickle=False) as z:
        if str(z['dataset_sha256'].item())!=DATASET_SHA:raise RuntimeError('EOF数据集哈希不匹配')
        if str(z['train_cut'].item())!=str(f['train_cut'].date()):raise RuntimeError('EOF训练截断日期不一致')
        if not(np.array_equal(z['lat1'],lat1) and np.array_equal(z['lon2'],lon2)):raise RuntimeError('EOF空间网格不一致')
        pp1=FrozenEOF(z['b1_mu'],z['b1_sd'],z['b1_V'])
        pp2=FrozenEOF(z['b2_mu'],z['b2_sd'],z['b2_V'])
    return dict(pp1=pp1,pp2=pp2,z1=pp1.z(F1),z2=pp2.z(F2),p1=pp1.pc(F1),p2=pp2.pc(F2),eof_sha=hashlib.sha256(raw).hexdigest())
fd=make_fold_data(FOLDS[0]);assert fd['pp1'].n>=1 and fd['pp2'].n>=1
print('Loaded frozen EOF:',fd['pp1'].n,fd['pp2'].n,'(per-fold independent)')

In [ ]:
# 5｜R6.5 原版联合双分支 Dataset + R6.1 物理距离损失；所有参数固定
class JointPairDataset(Dataset):
    def __init__(self,issues,fd):
        self.issues=np.asarray(issues,np.int64)
        self.z1=fd['z1'];self.z2=fd['z2'];self.p1=fd['p1'];self.p2=fd['p2']
    def __len__(self):return len(self.issues)
    def __getitem__(self,j):
        i=int(self.issues[j])
        def pack(z):
            v=z[i-H+1:i+1]
            d=np.empty_like(v);d[0]=0;d[1:]=v[1:]-v[:-1]
            return torch.from_numpy(np.concatenate([v,d],axis=-1).copy())
        y1=np.stack([self.p1[i+d] for d in c.LEADS]);y2=np.stack([self.p2[i+d] for d in c.LEADS])
        return pack(self.z1),pack(self.z2),torch.from_numpy(y1.copy()),torch.from_numpy(y2.copy())

def r61_loss(pred,target,fd,branch):
    # exact original R61 differentiable km-backprojection loss
    pc_mse=F.mse_loss(pred.float(),target.float())
    pp=fd[f'pp{branch}']
    V=torch.as_tensor(pp.V,dtype=torch.float32,device=pred.device)
    sd=torch.as_tensor(pp.sd,dtype=torch.float32,device=pred.device)
    scale=(torch.as_tensor(111.32*np.cos(np.deg2rad(lat1)),dtype=torch.float32,device=pred.device)
           if branch==1 else torch.ones_like(sd)*111.32)
    km=((pred.float()-target.float())@V.T)*((sd*scale)[None,None,:])
    phy=F.smooth_l1_loss(km/c.KM_SCALE,torch.zeros_like(km),beta=c.HUBER_BETA)
    return (1-c.ALPHA)*pc_mse+c.ALPHA*phy

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def nparams(m):return sum(int(p.numel()) for p in m.parameters())

In [ ]:
# 6｜后报评分：保留 R6.5 双向最近线段距离定义，同时报告日期及双分支物理坐标误差
# 该指标用于 GLORYS 50m 派生路径后报；不能与 ADT 参考误差直接视为同一真值。
def merge_path(b1coord,b2coord):
    a=np.asarray(b1coord,float);b=np.asarray(b2coord,float)
    candidates=np.flatnonzero(lon2>a[-1])
    k=int(candidates[np.argmin((lon2[candidates]-a[-1])**2+(b[candidates]-lat1[-1])**2)]) if len(candidates) else len(lon2)
    x=np.r_[a,lon2[k:]];y=np.r_[lat1,b[k:]]
    mask=np.isfinite(x)&np.isfinite(y)
    return x[mask],y[mask]

def one_direction(A,B):
    if min(len(A),len(B))<2:return np.asarray([np.nan])
    tree=cKDTree(B);_,kidx=tree.query(A,k=min(3,len(B)))
    kidx=np.asarray(kidx)
    if kidx.ndim==1:kidx=kidx[:,None]
    segments=np.unique(np.clip(np.concatenate([kidx.ravel()-1,kidx.ravel()]),0,len(B)-2))
    v=B[segments+1]-B[segments]
    t=np.clip(np.sum((A[:,None,:]-B[segments][None,:,:])*v[None,:,:],axis=2)/np.maximum(np.sum(v*v,axis=1),1e-12)[None,:],0,1)
    closest=B[segments][None,:,:]+t[:,:,None]*v[None,:,:]
    return np.sqrt(((A[:,None,:]-closest)**2).sum(axis=-1)).min(axis=1)

def errors_for_pair(p1,p2,y1,y2):
    px,py=merge_path(p1,p2);tx,ty=merge_path(y1,y2)
    lat0=float(np.mean(np.r_[py,ty]));lon0=float(np.mean(np.r_[px,tx]))
    cosine=np.cos(np.deg2rad(lat0))
    P=np.column_stack(((px-lon0)*111.32*cosine,(py-lat0)*111.32))
    T=np.column_stack(((tx-lon0)*111.32*cosine,(ty-lat0)*111.32))
    d=np.r_[one_direction(P,T),one_direction(T,P)]
    return float(np.mean(d**2)),float(np.mean(d)),float(np.percentile(d,90))

def score_predictions(f,seed,b1pred,b2pred):
    out=[]
    for j,i in enumerate(f['test']):
        for k,d in enumerate(c.LEADS):
            mse,mae,p90=errors_for_pair(b1pred[j,k],b2pred[j,k],A1[i+d],A2[i+d])
            if not np.isfinite(mse):raise RuntimeError('非有限预测路径评分')
            diff1=(b1pred[j,k]-A1[i+d])*111.32*np.cos(np.deg2rad(lat1))
            diff2=(b2pred[j,k]-A2[i+d])*111.32
            out.append(dict(test_year=int(f['year']),seed=int(seed),model=PAPER_NAME,issue_date=str(cal[i].date()),
                            target_date=str(cal[i+d].date()),lead_days=int(d),
                            path_mse_km2=mse,path_rmse_km=math.sqrt(mse),path_mae_km=mae,path_p90_km=p90,
                            branch1_km_rmse=float(np.sqrt(np.mean(diff1**2))),branch2_km_rmse=float(np.sqrt(np.mean(diff2**2))),
                            previously_development_year=int(f['year']) in (2014,2019)))
    return pd.DataFrame(out)

In [ ]:
# 5. 验证评价函数与R7.5的逐样本数值完全一致：真实冻结预测文件抽检
verified=False
try:
    yr=int(c.YEARS[0]);sd=int(c.SEEDS[0]);f=FOLDS[0]
    raw=get_r75_bytes(f'forecast_arrays/pred_{yr}_seed{sd}.npz')
    with np.load(BytesIO(raw),allow_pickle=False) as zz:
        pred1=np.asarray(zz['pred_b1_lon']);pred2=np.asarray(zz['pred_b2_lat']);issues=np.asarray(zz['issue_date'])
        assert str(zz['dataset_sha256'].item())==DATASET_SHA
    assert np.array_equal(issues,cal[f['test']].strftime('%Y-%m-%d'))
    for j in [0,min(4,len(issues)-1)]:
        for k,lead in enumerate(c.LEADS):
            ii=int(f['test'][j]); mse,mae,p90=errors_for_pair(pred1[j,k],pred2[j,k],A1[ii+lead],A2[ii+lead])
            row=R75_ERRORS[(R75_ERRORS.test_year==yr)&(R75_ERRORS.seed==sd)&(R75_ERRORS.issue_date==str(cal[ii].date()))&(R75_ERRORS.lead_days==lead)]
            assert len(row)==1
            if abs(float(row.iloc[0].path_mse_km2)-mse)>1e-3 or abs(float(row.iloc[0].path_mae_km)-mae)>1e-3:
                raise RuntimeError('R7.5 路径评分函数未复现，必须停止训练')
    verified=True
    print('[PASS] Frozen R75 forecast arrays produce exactly the same path MSE / MAE')
except FileNotFoundError:
    print('[WARN] R75归档中没有附带预测NPZ，无法执行数值逐样本交叉复核；但仍已使用R75原评分函数源代码')
if not verified: (REP/'R76_R75_metric_parity_warning.txt').write_text('R75 forecast arrays were missing; metric source was reused but numerical parity was not verified.',encoding='utf8')

In [ ]:
# 6. Persistence与Climatology：仅从训练期历史记录构建，无任何未来信息
# 气候态按月日映射到366天日历，2月29独立留位；各目标日期用训练期局地平滑。
LEAP_YEAR=2000

def climatological_doy(dt):
    d=pd.Timestamp(dt)
    return (pd.Timestamp(LEAP_YEAR,d.month,d.day)-pd.Timestamp(LEAP_YEAR,1,1)).days

def clim_maker(f):
    ix=np.flatnonzero(valid & (cal<f['train_cut']))
    day=np.asarray([climatological_doy(d) for d in cal[ix]])
    days=np.arange(366)
    dist=np.abs(day[:,None]-days[None,:]);dist=np.minimum(dist,366-dist)
    W=(dist<=c.CLIM_WINDOW_DAYS).astype(np.float64)
    counts=W.sum(axis=0)
    if np.any(counts==0):raise RuntimeError('clim calendar day no historical members')
    A=W.T@A1[ix]/counts[:,None]
    B=W.T@A2[ix]/counts[:,None]
    return A.astype(np.float32),B.astype(np.float32),counts

def persistence_predict(f):
    return np.repeat(F1[f['test'],None,:],3,axis=1),np.repeat(F2[f['test'],None,:],3,axis=1)

def climatology_predict(f):
    A,B,count=clim_maker(f)
    d=np.asarray([[climatological_doy(cal[int(i)+lead]) for lead in c.LEADS] for i in f['test']])
    return A[d],B[d],dict(min_history_count=float(np.min(count)),max_history_count=float(np.max(count)))

# 这两类确定性基线没有seed，输出时仅复制给两个seed作配对（不能视为独立样本）

In [ ]:
# 7. AR-EOF：直接预测领先EOF主成分，滞后阶/岭惩罚只在validation选择。
# 非领先PC同样作为输出目标，不丢弃高阶EOF结构。
def ar_design(pcs,issues,lag,K):
    idx=np.asarray(issues,dtype=np.int64)[:,None]-np.arange(lag)[None,:]
    if np.any(idx<0):raise RuntimeError('AR history precedes record')
    return np.asarray(pcs[idx,:K].reshape(len(issues),lag*K),np.float64)

class MultiRidge:
    def __init__(self,alpha):self.alpha=float(alpha)
    def fit(self,x,y):
        x=np.asarray(x,np.float64);y=np.asarray(y,np.float64)
        self.mx=x.mean(0);self.sx=x.std(0);self.sx[self.sx<1e-6]=1.
        self.my=y.mean(0);X=(x-self.mx)/self.sx;Y=y-self.my
        # Dense ridge, solve with Cholesky over feature Gram, multi-target simultaneously.
        G=X.T@X;G.flat[::G.shape[0]+1]+=self.alpha
        self.W=cho_solve(cho_factor(G,lower=True,check_finite=False),X.T@Y,check_finite=False)
        return self
    def predict(self,x):return ((np.asarray(x,np.float64)-self.mx)/self.sx@self.W+self.my).astype(np.float32)

def pack_targets(pc,issues):return np.stack([pc[issues+d] for d in c.LEADS],axis=1)

def physical_validation_loss(pred,target,pp,branch):
    pred=np.asarray(pred,np.float64);target=np.asarray(target,np.float64)
    mse=np.mean((pred-target)**2)
    factor=111.32*np.cos(np.deg2rad(lat1)) if branch==1 else np.ones_like(pp.sd)*111.32
    phys=((pred-target)@pp.V.T)*(pp.sd*factor)[None,None,:]/c.KM_SCALE
    z=np.abs(phys);beta=c.HUBER_BETA
    huber=np.mean(np.where(z<beta,.5*z*z/beta,z-.5*beta))
    return (1-c.ALPHA)*mse+c.ALPHA*huber

def ar_predict_branch(f,fd,branch):
    pc=fd[f'p{branch}'];pp=fd[f'pp{branch}']
    K=min(pc.shape[1],c.AR_INPUT_MAX_PCS)
    train_target=pack_targets(pc,f['train']).reshape(len(f['train']),-1)
    val_target=pack_targets(pc,f['val'])
    trials=[];best=None
    for lag in c.AR_LAGS:
        Xtr=ar_design(pc,f['train'],lag,K)
        Xva=ar_design(pc,f['val'],lag,K)
        for alpha in c.AR_ALPHAS:
            m=MultiRidge(alpha).fit(Xtr,train_target)
            pred=m.predict(Xva).reshape(len(f['val']),3,-1)
            score=float(physical_validation_loss(pred,val_target,pp,branch))
            trials.append({'year':f['year'],'branch':branch,'lag':lag,'alpha':alpha,'val_physical_loss':score,'input_pcs':K})
            if best is None or score<best[0]:best=(score,lag,alpha)
    _,lag,alpha=best
    m=MultiRidge(alpha).fit(ar_design(pc,f['train'],lag,K),train_target)
    out=m.predict(ar_design(pc,f['test'],lag,K)).reshape(len(f['test']),3,-1)
    return pp.invert(out),dict(branch=branch,selected_lag=lag,selected_alpha=alpha,selected_input_pcs=K,best_validation_loss=float(best[0]),trials=trials)

def ar_predict(f,fd):
    p1,m1=ar_predict_branch(f,fd,1);p2,m2=ar_predict_branch(f,fd,2)
    return p1,p2,[m1,m2]

In [ ]:
# 8. 同规模LSTM：不使用TCN/Conv/跨分支exchange；和主模型相同输入、loss与训练预算。
class OneBranchLSTM(nn.Module):
    def __init__(self,feature,hidden,pcdim):
        super().__init__();self.hidden=hidden;self.pcdim=pcdim
        self.rnn=nn.LSTM(feature,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pcdim)
    def forward(self,x):
        xin=F.dropout(x,p=c.DROPOUT,training=self.training)
        z,_=self.rnn(xin)
        return self.fc(z[:,-1]).reshape(len(x),3,self.pcdim)

class TwoBranchMatchedLSTM(nn.Module):
    def __init__(self,fd,h1,h2):
        super().__init__()
        self.net1=OneBranchLSTM(2*fd['z1'].shape[1],h1,fd['pp1'].n)
        self.net2=OneBranchLSTM(2*fd['z2'].shape[1],h2,fd['pp2'].n)
    def forward(self,x1,x2):return self.net1(x1),self.net2(x2)

def lstm_formula(inputdim,pdim,h):return 4*h*(inputdim+h+2)+3*pdim*(h+1)

def matched_lstm(fd,target):
    f1=2*fd['z1'].shape[1];f2=2*fd['z2'].shape[1];p1=fd['pp1'].n;p2=fd['pp2'].n
    h1=96 # pre-fixed branch allocation, do not optimize using test results
    h2=min(range(32,768),key=lambda h:abs(lstm_formula(f1,p1,h1)+lstm_formula(f2,p2,h)-target))
    model=TwoBranchMatchedLSTM(fd,h1,h2)
    actual=nparams(model)
    if abs(actual-target)/target>.01:raise RuntimeError(f'同规模LSTM参数不匹配 {actual} vs {target}')
    return model,dict(h1=h1,h2=h2,hybrid_parameters=target,lstm_parameters=actual,relative_delta=(actual-target)/target)

def status_path(f,seed):return WEIGHTS/str(f['year'])/str(seed)/'status.json'

def contract_lstm(f,fd,seed,param,target):
    obs={'type':'R76-MatchedLSTM-v1','paper':PAPER_NAME,'baseline':'Parameter-matched LSTM',
         'dataset_sha256':DATASET_SHA,'r75_eof_sha256':fd['eof_sha'],
         'test_year':f['year'],'seed':seed,'train_first':str(cal[f['train'][0]].date()),
         'train_last':str(cal[f['train'][-1]].date()),'val_last':str(cal[f['val'][-1]].date()),
         'test_issues_hash':hashlib.sha256('\n'.join(cal[f['test']].strftime('%Y-%m-%d')).encode('utf8')).hexdigest(),
         'npcs':(fd['pp1'].n,fd['pp2'].n),'hidden':(param['h1'],param['h2']),
         'same_loss':(c.ALPHA,c.KM_SCALE,c.HUBER_BETA),'optimizer':(c.LR,c.WEIGHT_DECAY),
         'epoch_cap':c.EPOCHS,'patience':c.PATIENCE,'dropout':c.DROPOUT,'matched_params':target}
    return hashlib.sha256(json.dumps(obs,sort_keys=True,default=str).encode()).hexdigest(),obs

def matched_lstm_train(f,fd,seed,target):
    seed_all(seed)
    model,params=matched_lstm(fd,target)
    model=model.to(DEV)
    cid,contract=contract_lstm(f,fd,seed,params,target)
    folder=WEIGHTS/str(f['year'])/str(seed);folder.mkdir(parents=True,exist_ok=True)
    bestfile=folder/'best.pt';latest=folder/'latest.pt';statusfile=folder/'status.json'
    if not c.FORCE_RETRAIN and bestfile.is_file() and statusfile.is_file():
        meta=json.loads(statusfile.read_text(encoding='utf8'))
        if meta.get('cid')==cid and meta.get('state')=='complete':
            ck=torch.load(bestfile,map_location='cpu',weights_only=True)
            if ck['cid']==cid:
                model.load_state_dict(ck['state'],strict=True);model.eval()
                return model,meta
    tr=DataLoader(JointPairDataset(f['train'],fd),batch_size=c.BATCH_SIZE,shuffle=True,num_workers=c.NUM_WORKERS)
    va=DataLoader(JointPairDataset(f['val'],fd),batch_size=c.BATCH_SIZE,shuffle=False,num_workers=c.NUM_WORKERS)
    opt=torch.optim.AdamW(model.parameters(),lr=c.LR,weight_decay=c.WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,patience=4,factor=.5)
    amp=(DEV.type=='cuda' and bool(c.USE_AMP))
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    epoch_first=1;best=float('inf');bad=0;hist=[];elapsed_prev=0.
    if latest.is_file() and not c.FORCE_RETRAIN:
        ck=torch.load(latest,map_location='cpu',weights_only=True)
        if ck.get('cid')!=cid:raise RuntimeError('latest.pt 合同已变化；请使用新输出目录')
        model.load_state_dict(ck['state'],strict=True)
        opt.load_state_dict(ck['opt']);sch.load_state_dict(ck['sch']);scaler.load_state_dict(ck['scaler'])
        epoch_first=int(ck['epoch'])+1;best=float(ck['best']);bad=int(ck['bad']);hist=list(ck['hist'])
        elapsed_prev=float(ck['elapsed_s'])
        if 'rng_torch' in ck:torch.set_rng_state(ck['rng_torch'])
        if DEV.type=='cuda' and ck.get('rng_cuda'):torch.cuda.set_rng_state_all(ck['rng_cuda'])
        print('[R76 RESUME]',f['year'],seed,'epoch',epoch_first)
    begin=time.perf_counter()
    for epoch in range(epoch_first,c.EPOCHS+1):
        if bad>=c.PATIENCE:break
        model.train();trloss=0.;trn=0
        for x1,x2,y1,y2 in tr:
            x1,x2,y1,y2=[t.to(DEV) for t in (x1,x2,y1,y2)]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda' if amp else 'cpu',enabled=amp):
                a,b=model(x1,x2)
                loss=.5*(r61_loss(a.float(),y1.float(),fd,1)+r61_loss(b.float(),y2.float(),fd,2))
            if not torch.isfinite(loss):raise RuntimeError('LSTM loss nonfinite')
            scaler.scale(loss).backward();scaler.unscale_(opt);nn.utils.clip_grad_norm_(model.parameters(),1.)
            scaler.step(opt);scaler.update();trloss+=float(loss.detach())*len(x1);trn+=len(x1)
        model.eval();valsum=0.;van=0
        with torch.inference_mode():
            for x1,x2,y1,y2 in va:
                x1,x2,y1,y2=[t.to(DEV) for t in (x1,x2,y1,y2)]
                with torch.autocast(device_type='cuda' if amp else 'cpu',enabled=amp):
                    a,b=model(x1,x2)
                    v=.5*(r61_loss(a.float(),y1.float(),fd,1)+r61_loss(b.float(),y2.float(),fd,2))
                valsum+=float(v)*len(x1);van+=len(x1)
        v=valsum/max(1,van);sch.step(v)
        if v<best-1e-7:
            best=v;bad=0
            torch.save({'cid':cid,'state':{k:t.detach().cpu() for k,t in model.state_dict().items()},'epoch':epoch,'best_val':v},bestfile)
        else:bad+=1
        hist.append({'test_year':f['year'],'seed':seed,'epoch':epoch,'train_loss':trloss/max(trn,1),
                     'val_loss':v,'best_val':best,'lr':float(opt.param_groups[0]['lr']),
                     'bad_epochs':bad,'elapsed_s':elapsed_prev+time.perf_counter()-begin})
        pd.DataFrame(hist).to_csv(folder/'train_history.csv',index=False)
        torch.save({'cid':cid,'state':{k:t.detach().cpu() for k,t in model.state_dict().items()},
                    'opt':opt.state_dict(),'sch':sch.state_dict(),'scaler':scaler.state_dict(),
                    'epoch':epoch,'best':best,'bad':bad,'hist':hist,
                    'elapsed_s':elapsed_prev+time.perf_counter()-begin,
                    'rng_torch':torch.get_rng_state(),
                    'rng_cuda':torch.cuda.get_rng_state_all() if DEV.type=='cuda' else []},latest)
        if epoch==1 or epoch%5==0 or bad>=c.PATIENCE:
            print('[R76 LSTM]',f['year'],seed,epoch,'train=',round(trloss/max(trn,1),4),'val=',round(v,4),'bad=',bad)
    if not bestfile.is_file():raise RuntimeError('LSTM无最佳权重')
    ck=torch.load(bestfile,map_location='cpu',weights_only=True)
    assert ck['cid']==cid
    model.load_state_dict(ck['state'],strict=True);model.eval()
    meta={'state':'complete','test_year':f['year'],'seed':seed,'cid':cid,
          'hidden1':params['h1'],'hidden2':params['h2'],'parameter_match_fraction':params['relative_delta'],
          'matched_model_parameters':params['lstm_parameters'],'hybrid_model_parameters':target,
          'best_epoch':int(ck['epoch']),'best_val':float(ck['best_val']),
          'elapsed_train_seconds':float(elapsed_prev+time.perf_counter()-begin)}
    statusfile.write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf8')
    return model,meta

def lstm_predict(model,fd,issues):
    loader=DataLoader(JointPairDataset(issues,fd),batch_size=c.BATCH_SIZE,shuffle=False,num_workers=c.NUM_WORKERS)
    out1=[];out2=[];model.eval()
    with torch.inference_mode():
        for x1,x2,_,_ in loader:
            a,b=model(x1.to(DEV),x2.to(DEV));out1.append(a.float().cpu().numpy());out2.append(b.float().cpu().numpy())
    return fd['pp1'].invert(np.concatenate(out1)),fd['pp2'].invert(np.concatenate(out2))

In [ ]:
# 9. preflight: 检查训练/目标、四基线形状与同规模参数量；失败不得开正式训练
fd=make_fold_data(FOLDS[0]);f=FOLDS[0];seed_all(9)
expected=int(R75_STATUS[R75_STATUS.year==f['year']].model_parameters.iloc[0])
m,par=matched_lstm(fd,expected)
assert abs(par['relative_delta'])<=.01
u=JointPairDataset(f['train'][:2],fd)
a,b,ya,yb=[x.unsqueeze(0) for x in u[0]]
m.train();q1,q2=m(a,b);loss=.5*(r61_loss(q1,ya,fd,1)+r61_loss(q2,yb,fd,2))
loss.backward();assert any(v.grad is not None for v in m.parameters())
m.eval();one=m(a,b)[0].detach().clone();two=m(a,b)[0].detach()
assert torch.allclose(one,two)
p1,p2=persistence_predict(f);cl1,cl2,info=climatology_predict(f)
assert p1.shape==(len(f['test']),3,len(lat1)) and cl2.shape==(len(f['test']),3,len(lon2))
assert np.isfinite(cl1).all() and np.isfinite(p2).all()
x=ar_design(fd['p2'],f['train'][:3],3,min(fd['pp2'].n,c.AR_INPUT_MAX_PCS))
assert x.shape[0]==3
print('[PASS] baseline mechanics and causal datasets; LSTM params',par,'| GPU',DEV)
del m,fd,loss;gc.collect()
if DEV.type=='cuda':torch.cuda.empty_cache()

In [ ]:
# 10. 四类基线正式执行：每折完成即时CSV、预测NPZ、状态，支持断点续跑。
# LSTM两个seed独立训练；其余3种为确定性基线，只执行1次再复制到两个seed作成对比较。
def score_baseline(f,seed,name,p1,p2):
    d=score_predictions(f,seed,p1,p2)
    d['model']=name
    return d

def save_preds(f,seed,name,p1,p2,cid):
    namekey={'Persistence':'persistence','Climatology':'climatology','AR-EOF':'ar_eof','Parameter-matched LSTM':'lstm'}[name]
    dest=PRED/f'pred_{namekey}_{f["year"]}_seed{seed}.npz'
    np.savez_compressed(dest,issue_date=np.asarray(cal[f['test']].strftime('%Y-%m-%d'),dtype='U10'),
                        target_date=np.asarray(np.column_stack([cal[f['test']+d].strftime('%Y-%m-%d') for d in c.LEADS]),dtype='U10'),
                        pred_b1_lon=p1,pred_b2_lat=p2,lead_days=np.asarray(c.LEADS),
                        cid=cid,dataset_sha256=DATASET_SHA,model=name)
    return dest

def load_cache(f,seed,name,cid):
    key={'Persistence':'persistence','Climatology':'climatology','AR-EOF':'ar_eof','Parameter-matched LSTM':'lstm'}[name]
    pred=PRED/f'pred_{key}_{f["year"]}_seed{seed}.npz'
    csv=TB/f'errors_{key}_{f["year"]}_seed{seed}.csv'
    if not(pred.is_file() and csv.is_file() and not c.FORCE_RETRAIN):return None
    try:
        with np.load(pred,allow_pickle=False) as z:
            if z['cid'].item()!=cid or z['dataset_sha256'].item()!=DATASET_SHA:return None
            if not np.array_equal(z['issue_date'],np.asarray(cal[f['test']].strftime('%Y-%m-%d'))):return None
            if not np.array_equal(z['lead_days'],np.asarray(c.LEADS)):return None
        df=pd.read_csv(csv)
        if len(df)!=len(f['test'])*len(c.LEADS) or not df.model.eq(name).all():return None
        return df
    except Exception:return None

def persist_result(f,seed,name,cid,p1,p2):
    key={'Persistence':'persistence','Climatology':'climatology','AR-EOF':'ar_eof','Parameter-matched LSTM':'lstm'}[name]
    df=score_baseline(f,seed,name,p1,p2)
    save_preds(f,seed,name,p1,p2,cid)
    df.to_csv(TB/f'errors_{key}_{f["year"]}_seed{seed}.csv',index=False)
    return df

def small_cid(f,fd,name,extra=''):
    z=f'{DATASET_SHA}|{fd["eof_sha"]}|{f["year"]}|{name}|{extra}|v1'
    return hashlib.sha256(z.encode()).hexdigest()

AR_SELECTION=[];LSTM_STATUS=[];ERR_PARTS=[];STOPPED=[]
if c.PREFLIGHT_ONLY or not c.RUN_FULL:
    print('[R76] PREFLIGHT_ONLY=True or RUN_FULL=False：正式实验未开始。')
else:
    for n,f in enumerate(FOLDS,1):
        print(f'\n==== Fold {n}/{len(FOLDS)} | {f["year"]} ====')
        fd=make_fold_data(f)
        for name in B_NAMES[:3]:
            cid=small_cid(f,fd,name,repr((c.AR_LAGS,c.AR_ALPHAS,c.AR_INPUT_MAX_PCS)) if name=='AR-EOF' else str(c.CLIM_WINDOW_DAYS))
            seed=int(c.SEEDS[0])
            cached=load_cache(f,seed,name,cid)
            if cached is not None:
                print('[CACHE]',name,f['year']);df=cached
            else:
                if name=='Persistence':p1,p2=persistence_predict(f);extra=None
                elif name=='Climatology':p1,p2,extra=climatology_predict(f)
                else:p1,p2,extra=ar_predict(f,fd)
                df=persist_result(f,seed,name,cid,p1,p2)
                if name=='AR-EOF':
                    AR_SELECTION.extend([{**r,'branch':e['branch']} for e in extra for r in e['trials']])
                    (REP/f'ar_selection_{f["year"]}.json').write_text(json.dumps(extra,ensure_ascii=False,indent=2),encoding='utf8')
                if name=='Climatology':
                    (REP/f'climatology_counts_{f["year"]}.json').write_text(json.dumps(extra),encoding='utf8')
            ERR_PARTS.append(df)
            for seed in c.SEEDS[1:]:
                seed=int(seed)
                # 对确定性基线的两个seed只是标签复制，且和R75精确同日期成对
                clone=df.copy();clone['seed']=seed
                ERR_PARTS.append(clone)
                key={'Persistence':'persistence','Climatology':'climatology','AR-EOF':'ar_eof'}[name]
                clone.to_csv(TB/f'errors_{key}_{f["year"]}_seed{seed}.csv',index=False)
        name='Parameter-matched LSTM'
        target=int(R75_STATUS[R75_STATUS.year==f['year']].model_parameters.iloc[0])
        for seed in c.SEEDS:
            seed=int(seed)
            probe,params=matched_lstm(fd,target);cid,_=contract_lstm(f,fd,seed,params,target);del probe
            cached=load_cache(f,seed,name,cid)
            if cached is not None and status_path(f,seed).is_file():
                meta=json.loads(status_path(f,seed).read_text())
                if meta.get('cid')==cid and meta.get('state')=='complete':
                    print('[CACHE] LSTM',f['year'],seed);ERR_PARTS.append(cached);LSTM_STATUS.append(meta);continue
            m,meta=matched_lstm_train(f,fd,seed,target)
            st=time.perf_counter();p1,p2=lstm_predict(m,fd,f['test']);elapsed=time.perf_counter()-st
            df=persist_result(f,seed,name,cid,p1,p2)
            meta['test_inference_seconds']=float(elapsed)
            status_path(f,seed).write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf8')
            ERR_PARTS.append(df);LSTM_STATUS.append(meta)
            print('[R76 LSTM TEST]',f['year'],seed,df.groupby('lead_days').path_mse_km2.mean().pow(.5).round(2).to_dict())
            del m,p1,p2,df;gc.collect()
            if DEV.type=='cuda':torch.cuda.empty_cache()
        del fd;gc.collect()
        pd.concat(ERR_PARTS,ignore_index=True).to_csv(TB/'R76_partial_baseline_errors.csv',index=False)
        pd.DataFrame(LSTM_STATUS).to_csv(TB/'R76_lstm_train_status.csv',index=False)
        print('[R76 YEAR FINISHED]',f['year'])
    pd.concat(ERR_PARTS,ignore_index=True).to_csv(TB/'R76_all_baseline_errors.csv',index=False)

In [ ]:
# 11. 盲于测试结果的正式对比：相同年份、种子、日期和+30/60/90目标
from collections import defaultdict

def paired_bootstrap(ref,compare,rep=400,block=30,seed=901):
    # ref, compare columns [test_year,issue_date,seed,path_mse_km2] in an individual lead.
    rng=np.random.default_rng(seed)
    aa=ref.merge(compare,on=['test_year','issue_date','seed'],suffixes=('_hybrid','_base'),validate='one_to_one')
    years=sorted(aa.test_year.unique())
    data=[]
    for y in years:
        dd=aa[aa.test_year==y].groupby('issue_date')[['path_mse_km2_hybrid','path_mse_km2_base']].mean().sort_index().to_numpy()
        data.append(dd)
    vals=[]
    for _ in range(rep):
        yrsel=rng.integers(0,len(years),len(years))
        mse=[]
        for ix in yrsel:
            dd=data[ix];n=len(dd);idx=[]
            while len(idx)<n:
                j=rng.integers(0,n)
                idx.extend(((j+np.arange(block))%n).tolist())
            mse.append(dd[np.asarray(idx[:n])].mean(0))
        hh,bb=np.sqrt(np.mean(mse,axis=0));vals.append(bb-hh)
    return np.percentile(vals,[2.5,50,97.5]).tolist(),float(np.mean(np.asarray(vals)>0))

if not c.PREFLIGHT_ONLY and c.RUN_FULL:
    base=pd.read_csv(TB/'R76_all_baseline_errors.csv')
    assert base.model.nunique()==4 and len(base)==len(R75_ERRORS)*4
    keys=['test_year','seed','issue_date','target_date','lead_days']
    R75_ERRORS[keys]=R75_ERRORS[keys].astype({'test_year':int,'seed':int,'issue_date':str,'target_date':str,'lead_days':int})
    base[keys]=base[keys].astype({'test_year':int,'seed':int,'issue_date':str,'target_date':str,'lead_days':int})
    if base.duplicated(keys+['model']).any():raise RuntimeError('基线出现重复的测试日期/seed')
    for name in B_NAMES:
        sub=base[base.model==name]
        joined=R75_ERRORS.merge(sub,on=keys,validate='one_to_one',suffixes=('_hybrid','_baseline'))
        if len(joined)!=len(R75_ERRORS):raise RuntimeError(f'{name}: 预测日期未与R75全部配对')
        if not np.isfinite(sub.path_mse_km2).all():raise RuntimeError('存在非有限基线误差')
    scores=pd.concat([R75_ERRORS,base],ignore_index=True)
    scores.to_csv(TB/'R76_all_five_model_paired_errors.csv',index=False)
    rows=[]
    for (name,yr,lead),g in scores.groupby(['model','test_year','lead_days'],sort=False):
        rows.append({'model':name,'test_year':yr,'lead_days':lead,'issues':g.issue_date.nunique(),'seeds':g.seed.nunique(),
                     'year_pooled_rmse_km':float(np.sqrt(g.path_mse_km2.mean())),
                     'mean_daily_rmse_km':float(g.path_rmse_km.mean()),
                     'mean_daily_mae_km':float(g.path_mae_km.mean())})
    annual=pd.DataFrame(rows)
    annual.to_csv(TB/'R76_five_models_year_by_lead.csv',index=False)
    macro=[]
    for (name,lead),g in annual.groupby(['model','lead_days']):
        macro.append({'model':name,'lead_days':lead,'n_years':g.test_year.nunique(),
                      'equal_year_rmse_km':float(np.sqrt(np.mean(g.year_pooled_rmse_km**2))),
                      'arithmetic_mean_year_rmse_km':float(g.year_pooled_rmse_km.mean())})
    macro=pd.DataFrame(macro)
    macro.to_csv(TB/'R76_macro_metrics_all_years.csv',index=False)
    # Exact numeric parity with original R75 macro definition
    original=read_r75_csv('tables/06_multi_year_macro_metrics.csv')
    mm=macro[macro.model==PAPER_NAME].sort_values('lead_days')
    assert np.allclose(mm.equal_year_rmse_km.values,original.sort_values('lead_days').equal_year_weight_rmse_km.values,rtol=0,atol=1e-6)
    print('[PASS] Model macro metrics reproduce R7.5 to <1e-6 km')
    all_pairs=[]
    for name in B_NAMES:
        for lead in c.LEADS:
            hybrid=R75_ERRORS[R75_ERRORS.lead_days==lead][['test_year','seed','issue_date','path_mse_km2']]
            other=base[(base.model==name)&(base.lead_days==lead)][['test_year','seed','issue_date','path_mse_km2']]
            inter=hybrid.merge(other,on=['test_year','seed','issue_date'],suffixes=('_hybrid','_base'),validate='one_to_one')
            # Official R7.5 aggregation: first pooled MSE per year, then equal-year mean MSE, then sqrt.
            year_mse=inter.groupby('test_year')[['path_mse_km2_hybrid','path_mse_km2_base']].mean()
            delta=float(np.sqrt(year_mse.path_mse_km2_base.mean())-np.sqrt(year_mse.path_mse_km2_hybrid.mean()))
            delta_sample=float(np.sqrt(inter.path_mse_km2_base.mean())-np.sqrt(inter.path_mse_km2_hybrid.mean()))
            ci,pr=paired_bootstrap(hybrid,other,c.BOOT_REPS,c.BOOT_BLOCK_DAYS)
            all_pairs.append({'baseline':name,'lead_days':lead,'equal_year_delta_baseline_minus_hybrid_km':delta,'sample_pooled_delta_km':delta_sample,
                              'CI95_lo_km':ci[0],'CI95_hi_km':ci[2],'bootstrap_fraction_hybrid_better':pr,
                              'n_paired_seed_forecasts':len(inter),'winner_at_pooled':PAPER_NAME if delta>0 else name})
    pd.DataFrame(all_pairs).to_csv(TB/'R76_paired_year_block_bootstrap.csv',index=False)
    # 2014/2019 used in architectural development: separate analysis WITHOUT those years
    no_dev=annual[~annual.test_year.isin([2014,2019])]
    macro_nodev=no_dev.groupby(['model','lead_days']).agg(
        n_years=('test_year','nunique'),mean_sq=('year_pooled_rmse_km',lambda z:np.mean(np.asarray(z)**2))).reset_index()
    macro_nodev['equal_year_rmse_km']=np.sqrt(macro_nodev['mean_sq']);macro_nodev.drop(columns='mean_sq',inplace=True)
    macro_nodev.to_csv(TB/'R76_sensitivity_exclude_development_years.csv',index=False)
    # More conservative: the earlier R6.1 development chain also exposed 2011 as a QA year.
    strict_dev_years=[2011,2014,2019]
    strict_annual=annual[~annual.test_year.isin(strict_dev_years)]
    strict_macro=strict_annual.groupby(['model','lead_days']).agg(
        n_years=('test_year','nunique'),mean_sq=('year_pooled_rmse_km',lambda z:np.mean(np.asarray(z)**2))).reset_index()
    strict_macro['equal_year_rmse_km']=np.sqrt(strict_macro.pop('mean_sq'))
    strict_macro.to_csv(TB/'R76_conservative_exclude_2011_2014_2019.csv',index=False)
    print('\n### R76 FAIR BASELINE COMPARISON ###')
    display(macro.pivot(index='model',columns='lead_days',values='equal_year_rmse_km').round(3))
    display(pd.DataFrame(all_pairs).round(3))
    # plots
    plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False})
    fig,ax=plt.subplots(figsize=(9.2,4.7))
    for name in [PAPER_NAME,*B_NAMES]:
        ss=macro[macro.model==name].sort_values('lead_days')
        ax.plot(ss.lead_days,ss.equal_year_rmse_km,marker='o',label=name)
    ax.set_xlabel('Forecast lead (days)');ax.set_ylabel('Equal-year path RMSE (km)')
    ax.set_title('Rolling hindcast: hybrid deep-learning prediction vs. four baselines')
    ax.legend(fontsize=8,loc='best');ax.grid(alpha=.17);fig.tight_layout()
    fig.savefig(FIG/'Fig01_R76_Fair_Baseline_Comparison.png',dpi=c.FIG_DPI);fig.savefig(FIG/'Fig01_R76_Fair_Baseline_Comparison.pdf');plt.show();plt.close(fig)
    fig,axes=plt.subplots(1,3,figsize=(15,4.2),sharey=True)
    for ax,lead in zip(axes,c.LEADS):
        aa=annual[annual.lead_days==lead]
        for name in [PAPER_NAME,*B_NAMES]:
            d=aa[aa.model==name].sort_values('test_year');ax.plot(d.test_year,d.year_pooled_rmse_km,marker='.',label=name)
        ax.set_title(f'+{lead} days');ax.set_xlabel('Test year');ax.grid(alpha=.2)
    axes[0].set_ylabel('Year-wise path RMSE (km)');axes[-1].legend(fontsize=7);fig.tight_layout()
    fig.savefig(FIG/'Fig02_R76_Year_by_Year.png',dpi=c.FIG_DPI);fig.savefig(FIG/'Fig02_R76_Year_by_Year.pdf');plt.show();plt.close(fig)

In [ ]:
# 12. 最终归档：审稿论文图表、逐样本误差、AR参数选择、LSTM复杂度和自动下载
import zipfile

def archive(stage='completed'):
    done=[]
    for f in FOLDS:
        for seed in c.SEEDS:
            d=WEIGHTS/str(f['year'])/str(seed)/'status.json'
            if d.is_file():
                try:
                    s=json.loads(d.read_text(encoding='utf8'))
                    if s.get('state')=='complete':done.append([int(f['year']),int(seed)])
                except Exception:pass
    complete=len(done)==len(FOLDS)*len(c.SEEDS) and (TB/'R76_macro_metrics_all_years.csv').is_file()
    st={'stage':stage,'complete':bool(complete),'completed_lstm_seed_runs':len(done),
        'expected_lstm_seed_runs':len(FOLDS)*len(c.SEEDS),'model_name':PAPER_NAME,
        'dataset_sha256':DATASET_SHA,'r75_training_reused_without_modification':True,
        'r75_test_results_used_only_for_final_comparison':True,
        'all_baselines_share_exact_r75_issue_dates':True,'development_exposed_years_r75':[2014,2019],'additional_earlier_qa_year':[2011],
        'results_reference':'GLORYS 50m reconstructed paths; ADT evaluated separately'}
    (REP/'R76_status.json').write_text(json.dumps(st,indent=2,ensure_ascii=False),encoding='utf8')
    excludes={'.pt','.pth'}
    with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for folder in (TB,REP,FIG):
            for p in sorted(folder.rglob('*')):
                if p.is_file() and p.suffix not in excludes:
                    z.write(p,p.relative_to(RUN).as_posix())
        # local .pt weights and all forecast .npz remain local. Include representative seeds for easy review.
        for p in sorted(PRED.glob('pred_*_2005_seed71.npz'))[:4]:
            z.write(p,p.relative_to(RUN).as_posix())
        for p in sorted(WEIGHTS.glob('*/*/train_history.csv')):
            z.write(p,p.relative_to(RUN).as_posix())
    digest=sha256_file(ARCHIVE)
    Path(str(ARCHIVE)+'.sha256').write_text(digest+'  '+ARCHIVE.name+'\n',encoding='utf8')
    assert zipfile.ZipFile(ARCHIVE).testzip() is None
    print('ZIP:',ARCHIVE,'MB',round(ARCHIVE.stat().st_size/1e6,2),'sha256',digest,'complete',complete)
    if 'get_ipython' in globals() and get_ipython() is not None:
        display(FileLink(str(ARCHIVE)))
        display(FileLink(str(ARCHIVE)+'.sha256'))
    return st

status=archive('completed' if c.RUN_FULL and not c.PREFLIGHT_ONLY else 'preflight_only')

## 结果解读注意

- 必须同时报告 **每年等权 RMSE**、逐年度走势、配对差值 95% CI，不允许仅挑选 hybrid deep-learning prediction 获胜的年份。
- R7.5 训练期 EOF 和本轮两分支基线共享同一日期和数据预处理，但 LSTM 没有卷积或跨分支模块；参数量匹配消除了一个主要混杂，**并不代表计算量或归纳偏置相同**。
- AR-EOF 使用验证集选择 Ridge/滞后配置；Persistence、Climatology 没有可学习深度模型参数；LSTM 的两个种子为独立训练，确定性基线重复 seed 仅用于配对，不能计算为独立实验次数。
- R7.5 原报告标注 2014、2019 为研发暴露年份；考虑更早的 R6.1 研究中也使用过 2011 年 QA，额外输出更保守的排除 2011/2014/2019 分析。结果不能宣传为十年全盲测。
- 表中 GLORYS 50 m 路径误差不能混同 R7.4 的 ADT 独立参考流轴误差。